In [1]:
"""
Assignment 1: Reinforcement Learning from Human Feedback (RLHF)
Notebook: 05_ppo_helpsteer2.ipynb

Purpose:
This notebook performs the RLHF policy-optimisation stage using Proximal Policy
Optimisation (PPO). It starts from the SFT model and uses the reward model trained
on HelpSteer2-derived preference pairs to optimise the policy toward responses that
better match that reward signal.

Summary of sections:
1. Environment setup and package imports
   - Imports PPO / RLHF libraries and prepares the runtime environment.

2. Loading models and tokenizer
   - Loads the SFT model, tokenizer, and the HelpSteer2 reward model.

3. PPO dataset preparation
   - Loads and formats prompts used during PPO training.

4. PPO configuration
   - Defines PPO hyperparameters, generation settings, and optimisation settings.

5. PPO training loop
   - Generates responses, scores them using the HelpSteer2 reward model,
     and updates the policy through PPO.

6. Logging and monitoring
   - Records rewards, trends, and selected generated samples during training.

7. Saving outputs
   - Saves the PPO-trained HelpSteer2-based policy model/checkpoint.

8. Sample generation / inspection
   - Generates example outputs from the PPO-HelpSteer2 model for qualitative checking.

Main output:
- PPO-aligned model trained using the HelpSteer2-based reward model.
"""

'\nAssignment 1: Reinforcement Learning from Human Feedback (RLHF)\nNotebook: 05_ppo_helpsteer2.ipynb\n\nPurpose:\nThis notebook performs the RLHF policy-optimisation stage using Proximal Policy\nOptimisation (PPO). It starts from the SFT model and uses the reward model trained\non HelpSteer2-derived preference pairs to optimise the policy toward responses that\nbetter match that reward signal.\n\nSummary of sections:\n1. Environment setup and package imports\n   - Imports PPO / RLHF libraries and prepares the runtime environment.\n\n2. Loading models and tokenizer\n   - Loads the SFT model, tokenizer, and the HelpSteer2 reward model.\n\n3. PPO dataset preparation\n   - Loads and formats prompts used during PPO training.\n\n4. PPO configuration\n   - Defines PPO hyperparameters, generation settings, and optimisation settings.\n\n5. PPO training loop\n   - Generates responses, scores them using the HelpSteer2 reward model,\n     and updates the policy through PPO.\n\n6. Logging and moni

In [6]:
from google.colab import drive
import os

drive.mount('/content/drive')

SFT_ADAPTER_PATH = "/content/drive/MyDrive/rlhf_assignment_outputs/sft_qwen25_05b_base"
RM_HELPSTEER2_PATH = "/content/drive/MyDrive/rlhf_assignment_outputs/rm_helpsteer2"

print("SFT exists:", os.path.exists(SFT_ADAPTER_PATH))
print("RM_HELPSTEER2 exists:", os.path.exists(RM_HELPSTEER2_PATH))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SFT exists: True
RM_HELPSTEER2 exists: True


In [7]:
# Install packages

!pip uninstall -y trl transformers peft accelerate sympy -q
!pip install -q --no-cache-dir \
  "sympy==1.12" \
  "transformers==4.45.2" \
  "trl==0.11.4" \
  "peft==0.12.0" \
  "accelerate==0.34.2" \
  "datasets==2.21.0" \
  "evaluate==0.4.3" \
  "rouge_score" \
  "bert_score" \
  "scikit-learn"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 318.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 237.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 355.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 381.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 375.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.25.0+cu128 requires torch==2.10.0, but you have torch 2.4.1 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.4.1 which is incompatible.


In [10]:
# Import Libraries

import os
import gc
import json
import random
import warnings
import inspect
import shutil
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
)
from peft import PeftModel, LoraConfig
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer

warnings.filterwarnings("ignore")


# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')


# Version Check

import sympy
import peft
import transformers
import trl
import accelerate

print("sympy:", sympy.__version__)
print("peft version:", peft.__version__)
print("transformers version:", transformers.__version__)
print("trl version:", trl.__version__)
print("accelerate version:", accelerate.__version__)


# Model and Configuration

BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
SFT_ADAPTER_PATH = "/content/drive/MyDrive/rlhf_assignment_outputs/sft_qwen25_05b_base"

RM_HS2_PATH = "/content/drive/MyDrive/rlhf_assignment_outputs/rm_helpsteer2"

OUTPUT_DIR = "./outputs/ppo_helpsteer2"

SEED = 42
MAX_PROMPT_LENGTH = 256
MAX_NEW_TOKENS = 96
MAX_RM_LENGTH = 512
TRAIN_SAMPLES = 500
BATCH_SIZE = 4
MINI_BATCH_SIZE = 2
GRAD_ACCUM = 1
PPO_EPOCHS = 2
LEARNING_RATE = 1e-5

os.makedirs("./outputs", exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print("Device:", device)
print("Base model:", BASE_MODEL_NAME)
print("SFT adapter:", SFT_ADAPTER_PATH)
print("Reward model:", RM_HS2_PATH)
print("SFT exists:", os.path.exists(SFT_ADAPTER_PATH))
print("RM exists:", os.path.exists(RM_HS2_PATH))


# Show Saved Files

if os.path.exists(SFT_ADAPTER_PATH):
    print("\nFiles in SFT adapter path:")
    for f in os.listdir(SFT_ADAPTER_PATH):
        print(" ", f)

if os.path.exists(RM_HS2_PATH):
    print("\nFiles in RM_HS2 path:")
    for f in os.listdir(RM_HS2_PATH):
        print(" ", f)


# Helper Function: sanitize adapter config for current PEFT version

def sanitize_lora_config(config_path):
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"adapter_config.json not found at: {config_path}")

    with open(config_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    allowed_keys = set(inspect.signature(LoraConfig.__init__).parameters.keys())
    allowed_keys.discard("self")

    removed = {}
    clean_cfg = {}

    for k, v in cfg.items():
        if k in allowed_keys:
            clean_cfg[k] = v
        else:
            removed[k] = v

    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(clean_cfg, f, indent=2)

    return removed, clean_cfg


# Sanitize SFT adapter_config.json

sft_config_path = os.path.join(SFT_ADAPTER_PATH, "adapter_config.json")
removed_sft, clean_sft = sanitize_lora_config(sft_config_path)

print("\nSanitized SFT adapter_config.json")
print("Removed unsupported SFT keys:", list(removed_sft.keys()))
print("Remaining SFT keys:", list(clean_sft.keys()))


# Load tokenizer and PPO model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    if tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto"
)

sft_peft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_PATH)

ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(sft_peft_model)
ppo_model = ppo_model.to(device)

ref_base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto"
)

ref_peft_model = PeftModel.from_pretrained(ref_base_model, SFT_ADAPTER_PATH)
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(ref_peft_model)
ref_model = ref_model.to(device)

print("\nPPO policy and reference model loaded successfully.")


# Load reward model

rm_tokenizer = AutoTokenizer.from_pretrained(RM_HS2_PATH, use_fast=True)

if rm_tokenizer.pad_token is None:
    if rm_tokenizer.sep_token is not None:
        rm_tokenizer.pad_token = rm_tokenizer.sep_token
    elif rm_tokenizer.eos_token is not None:
        rm_tokenizer.pad_token = rm_tokenizer.eos_token
    else:
        rm_tokenizer.add_special_tokens({"pad_token": "[PAD]"})

reward_model = AutoModelForSequenceClassification.from_pretrained(
    RM_HS2_PATH
).to(device)

reward_model.eval()

print("Reward model loaded successfully.")


# Load HelpSteer2 prompts

raw = load_dataset("nvidia/HelpSteer2")
train_source = raw["train"].shuffle(seed=SEED).select(range(TRAIN_SAMPLES))

print("\nLoaded HelpSteer2 train subset:", len(train_source))
print("Columns:", train_source.column_names)
print(train_source[0])


# Helper Functions

def normalize_prompt(example):
    prompt = str(example.get("prompt", "")).strip()
    prompt = f"Instruction: {prompt}\n\nResponse:"
    return {"prompt": prompt}

def tokenize_prompt(example):
    encoded = tokenizer(
        example["prompt"],
        truncation=True,
        max_length=MAX_PROMPT_LENGTH,
        padding=False
    )
    return {
        "query": example["prompt"],
        "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long)
    }

def collator(features):
    return {k: [f[k] for f in features] for k in features[0]}

def build_full_text(prompt, response):
    prompt = str(prompt).strip()
    response = str(response).strip()

    if prompt.startswith("Instruction:") and prompt.endswith("Response:"):
        instruction_text = prompt.replace("Instruction:", "").replace("Response:", "").strip()
        return f"Human: {instruction_text}\n\nAssistant: {response}"

    return f"Human: {prompt}\n\nAssistant: {response}"

def score_texts(texts):
    inputs = rm_tokenizer(
        texts,
        truncation=True,
        max_length=MAX_RM_LENGTH,
        padding=True,
        return_tensors="pt"
    )

    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = reward_model(**inputs).logits.squeeze(-1)

    return logits.detach().cpu()

dataset = train_source.map(normalize_prompt)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "prompt"])
dataset = dataset.filter(lambda x: len(x["prompt"]) > 0)
dataset = dataset.map(tokenize_prompt)

print("\nProcessed PPO dataset size:", len(dataset))
print("Example prompt:")
print(dataset[0]["query"][:500])


# PPO Configuration

ppo_config = PPOConfig(
    model_name=BASE_MODEL_NAME,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    mini_batch_size=MINI_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    ppo_epochs=PPO_EPOCHS,
    seed=SEED,
    log_with=None,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=ppo_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    dataset=dataset,
    data_collator=collator
)

generation_kwargs = {
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": True,
    "top_p": 0.9,
    "temperature": 0.7,
    "repetition_penalty": 1.1,
    "pad_token_id": tokenizer.pad_token_id,
    "eos_token_id": tokenizer.eos_token_id,
}

print("\nPPOTrainer created successfully.")


# PPO Training Loop

training_logs = []
sample_outputs = []

for step, batch in enumerate(tqdm(ppo_trainer.dataloader, desc="PPO HelpSteer2 training")):
    query_tensors = batch["input_ids"]
    query_texts = batch["query"]

    query_tensors = [
        q if isinstance(q, torch.Tensor) else torch.tensor(q, dtype=torch.long)
        for q in query_tensors
    ]

    response_tensors = ppo_trainer.generate(
        query_tensors,
        return_prompt=False,
        **generation_kwargs
    )

    response_texts = tokenizer.batch_decode(response_tensors, skip_special_tokens=True)

    full_texts = [build_full_text(q, r) for q, r in zip(query_texts, response_texts)]
    reward_scores = score_texts(full_texts)

    rewards = [torch.tensor(float(x)) for x in reward_scores]

    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

    mean_reward = float(torch.mean(reward_scores).item())
    std_reward = float(torch.std(reward_scores).item()) if len(reward_scores) > 1 else 0.0
    mean_len = float(np.mean([len(tokenizer.encode(x, add_special_tokens=False)) for x in response_texts]))

    row = {
        "step": step,
        "mean_reward": mean_reward,
        "std_reward": std_reward,
        "mean_response_len": mean_len
    }

    for k, v in stats.items():
        try:
            row[k] = float(v) if not isinstance(v, (list, dict, tuple)) else str(v)
        except Exception:
            row[k] = str(v)

    training_logs.append(row)

    if step % 10 == 0:
        print("=" * 80)
        print("Step:", step)
        print("Mean reward:", mean_reward)
        print("Sample prompt:")
        print(query_texts[0][:500])
        print("Sample response:")
        print(response_texts[0][:500])

        sample_outputs.append({
            "step": step,
            "prompt": query_texts[0],
            "response": response_texts[0],
            "reward": float(reward_scores[0].item())
        })

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# Save PPO model and logs

ppo_trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

pd.DataFrame(training_logs).to_csv(f"{OUTPUT_DIR}/ppo_helpsteer2_logs.csv", index=False)

with open(f"{OUTPUT_DIR}/ppo_helpsteer2_samples.jsonl", "w", encoding="utf-8") as f:
    for item in sample_outputs:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("\nSaved PPO-HelpSteer2 model to:", OUTPUT_DIR)


# Copy PPO model to Google Drive

drive_output_root = "/content/drive/MyDrive/rlhf_assignment_outputs"
os.makedirs(drive_output_root, exist_ok=True)

drive_ppo_path = os.path.join(drive_output_root, "ppo_helpsteer2")

if os.path.exists(drive_ppo_path):
    shutil.rmtree(drive_ppo_path)

shutil.copytree(OUTPUT_DIR, drive_ppo_path)

print("Copied PPO-HelpSteer2 model to Drive:", drive_ppo_path)
print("Files in Drive PPO-HelpSteer2 folder:")
for f in os.listdir(drive_ppo_path):
    print(" ", f)


# Final sanity generation

test_prompt = "Instruction: Explain what makes an answer more helpful to a user.\n\nResponse:"
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = ppo_trainer.model.pretrained_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.1,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

print("=" * 80)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
sympy: 1.12
peft version: 0.12.0
transformers version: 4.45.2
trl version: 0.11.4
accelerate version: 0.34.2
Device: cuda
Base model: Qwen/Qwen2.5-0.5B-Instruct
SFT adapter: /content/drive/MyDrive/rlhf_assignment_outputs/sft_qwen25_05b_base
Reward model: /content/drive/MyDrive/rlhf_assignment_outputs/rm_helpsteer2
SFT exists: True
RM exists: True

Files in SFT adapter path:
  checkpoint-100
  checkpoint-125
  README.md
  adapter_model.safetensors
  tokenizer.json
  training_args.bin
  chat_template.jinja
  tokenizer_config.json
  adapter_config.json

Files in RM_HS2 path:
  README.md
  config.json
  model.safetensors
  tokenizer_config.json
  training_args.bin
  tokenizer.json
  checkpoint-32
  checkpoint-4

Sanitized SFT adapter_config.json
Removed unsupported SFT keys: []
Remaining SFT keys: ['alpha_pattern', 'auto_mapping', 'base_model_name_or_path', 'bias

PPO HelpSteer2 training:   0%|          | 0/125 [00:00<?, ?it/s]

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step: 0
Mean reward: 0.0314963199198246
Sample prompt:
Instruction: what is a spot VM?

Response:
Sample response:
 A spot virtual machine is an Amazon EC2 instance that's reserved for use by the customer and does not incur any cost until used. Spot instances are priced at a discounted rate compared to standard Amazon EC2 reservations, with no prepayment required.

Spot instances have no billing threshold and can be purchased in advance. The pricing of the EC2 service varies depending on the region where it is located. For example, the cheapest spot price in New York City (NYS) is $
Step: 10
Mean reward: 0.014140361919999123
Sample prompt:
Instruction: List all possible Neural Network Architectures with descriptions
<extra_id_1>Assistant
Sure, here are some examples of common Neural Network Architectures:
 

 -  Feedforward Neural Network
 -  Multi-Layer Perceptron
 -  Convolutional Neural Network
 -  Long Short-Term Memory (LSTM) Network
 -  Restricted Boltzmann Machine
 

 I could al